# OpenEnv Colab PR Flow Notebook

Use this notebook after implementation is done to standardize PR quality and reduce review churn.

In [ ]:
# Module A: Config
REPO_URL = "https://github.com/soham2710/OpenEnv.git"
BRANCH = "test/hf-deploy-script-coverage"
BASE_BRANCH = "main"
WORKDIR = "OpenEnv"
ISSUE_NUMBER = 54
PR_TITLE = "<fill me>"
PYTEST_TARGET = "tests/scripts/test_prepare_hf_deployment.py"
PYTEST_FLAGS = "-q"
RUN_RUFF = True

In [ ]:
# Module B: Bootstrap
import os
import subprocess
from pathlib import Path

def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, text=True)

workspace = Path('/content') / WORKDIR
if workspace.exists():
    run(f"rm -rf {workspace}")

run(f"git clone -b {BRANCH} {REPO_URL} {WORKDIR}")
os.chdir(workspace)
run("python -m pip install -q uv")
run("uv sync --group dev")
run("git fetch origin")
run("git status --short --branch")

In [ ]:
# Module C: Validation Gates
os.chdir(f"/content/{WORKDIR}")
run(f"PYTHONPATH=src:envs .venv/bin/uv run pytest {PYTEST_TARGET} {PYTEST_FLAGS}")

if RUN_RUFF:
    run(".venv/bin/uv run ruff check src/ tests/")

run("git diff --check")
print("Validation gates passed")

In [ ]:
# Module D: Risk Review Checklist
review_checks = [
    "No unrelated files changed",
    "Tests prove behavior change",
    "No cross-branch docs links",
    "No breaking API changes without docs",
    "No secrets/tokens in changes",
]
for i, item in enumerate(review_checks, start=1):
    print(f"[{i}] {item}")

print("Mark these as done in your PR description.")

In [ ]:
# Module E: Generate PR body artifact
import subprocess
from pathlib import Path

status = subprocess.check_output("git status --short --branch", shell=True, text=True)
changed = subprocess.check_output("git diff --name-only", shell=True, text=True)
diff_stat = subprocess.check_output("git diff --stat", shell=True, text=True)

pr_body = f"""## Summary
- <add concise bullet list of what changed>

## Why
- Resolves issue #{ISSUE_NUMBER}

## Validation
- PYTHONPATH=src:envs .venv/bin/uv run pytest {PYTEST_TARGET} {PYTEST_FLAGS}

## Changed files
```
{changed.strip() or 'No changed files'}
```

## Diff stat
```
{diff_stat.strip() or 'No diff stat'}
```

## Branch status
```
{status.strip()}
```

Refs #{ISSUE_NUMBER}
"""

artifact = Path("artifacts") / f"pr_body_issue_{ISSUE_NUMBER}.md"
artifact.parent.mkdir(parents=True, exist_ok=True)
artifact.write_text(pr_body, encoding="utf-8")
print(f"Wrote {artifact}")

## Module F: Publish

Run these in a terminal cell if needed:
- `git add <files>`
- `git commit -m "<message>"`
- `git push`

Then open a PR and paste the generated artifact from `artifacts/pr_body_issue_<n>.md`.